# 08 — Evaluation

Evaluate trained MS-ZeroGAD on all target datasets (zero-shot).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
PROJECT_ROOT = '/content/drive/MyDrive/Project_GraphML/ms-zerogad'
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

In [ ]:
import torch
import yaml
import json
import pandas as pd

from ms_zerogad.data.loader import load_graph_dataset
from ms_zerogad.data.preprocessing import sparse_to_torch_dense, feature_to_torch
from ms_zerogad.pipeline.multipass import MultiPassPipeline
from ms_zerogad.evaluation.metrics import evaluate_pipeline, evaluate_per_pass

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Load checkpoint

In [ ]:
ckpt_dir = 'checkpoints'
ckpt_path = os.path.join(ckpt_dir, 'best.pt')

if not os.path.exists(ckpt_path):
    raise FileNotFoundError(
        f'No best checkpoint found at {ckpt_path}. '
        f'Run 07_train.ipynb first with save_best_path enabled.'
    )

print(f'Loading: {ckpt_path}')
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
cfg = ckpt['config']

# In thông tin về checkpoint để verify
print(f'Best epoch: {ckpt.get("best_epoch", "?")}')
print(f'Best AUROC: {ckpt.get("best_auroc", "?"):.4f}' if 'best_auroc' in ckpt else 'Best AUROC: N/A')
print(f'Total epochs trained: {len(ckpt["history"]["epoch"])}')
print(f'Final training loss: {ckpt["history"]["total_loss"][-1]:.4f}')

In [ ]:
# Reconstruct pipeline
pipeline = MultiPassPipeline(
    d_prime=cfg['module1']['d_prime'],
    band_low=cfg['module1']['band_low'],
    band_high=cfg['module1']['band_high'],
    alpha_low=cfg['module1']['alpha_low'],
    alpha_mid=cfg['module1']['alpha_mid'],
    alpha_high=cfg['module1']['alpha_high'],
    k_smoothing=cfg['module2']['k_smoothing'],
    sigma=cfg['module2']['sigma'],
    D_rff=cfg['module2']['D_rff'],
    d_svd=cfg['module2']['d_svd'],
    tau=cfg['module2']['tau'],
    kmeans_max_iter=cfg['module2']['kmeans_max_iter'],
    d_hidden=cfg['module4']['d_hidden'],
    d_latent=cfg['module4']['d_latent'],
    num_encoder_layers=cfg['module4']['num_encoder_layers'],
    num_decoder_layers=cfg['module4']['num_decoder_layers'],
    dropout=cfg['module4']['dropout'],
).to(device)

pipeline.load_state_dict(ckpt['state_dict'])
pipeline.eval()
print('Loaded.')

## Evaluate on all target datasets

In [ ]:
target_names = cfg['datasets']['target']
rows = []

for name in target_names:
    path = f'/content/drive/MyDrive/Project_GraphML/ms-zerogad/ms_zerogad/data/raw/{name}.mat'
    if not os.path.exists(path):
        print(f'SKIP: {name} not found')
        continue

    A_sp, X_sp, y_np = load_graph_dataset(path)
    A = sparse_to_torch_dense(A_sp)
    X = feature_to_torch(X_sp, dense=True)
    y = torch.from_numpy(y_np).long()

    # Per-pass + aggregated
    results = evaluate_per_pass(pipeline, X, A, y, device=device)

    row = {'dataset': name, 'n': X.shape[0], 'anomalies': int(y.sum().item())}
    for key in ['pass1', 'pass2', 'pass3', 'aggregated']:
        row[f'{key}_auroc'] = results[key]['auroc']
        row[f'{key}_auprc'] = results[key]['auprc']
    rows.append(row)

    print(f'{name:<12} | AUROC: '
          f'P1={results["pass1"]["auroc"]:.3f} '
          f'P2={results["pass2"]["auroc"]:.3f} '
          f'P3={results["pass3"]["auroc"]:.3f} '
          f'AGG={results["aggregated"]["auroc"]:.3f}')

df = pd.DataFrame(rows)
df.to_csv('/content/drive/MyDrive/Project_GraphML/ms-zerogad/results/evaluation.csv', index=False)
df

## Summary table

In [ ]:
summary_cols = ['dataset', 'pass1_auroc', 'pass2_auroc', 'pass3_auroc', 'aggregated_auroc']
print('AUROC per pass and aggregated:')
print(df[summary_cols].to_string(index=False, float_format='%.4f'))
print(f'\nMean aggregated AUROC: {df["aggregated_auroc"].mean():.4f}')
print(f'Mean aggregated AUPRC: {df["aggregated_auprc"].mean():.4f}')